In [2]:
"""
NYT Sentiment Analysis — Chen (2026) LES Public Housing Paper
=============================================================
SETUP:  pip install requests vaderSentiment pandas matplotlib scipy tqdm
USAGE:  Set NYT_API_KEY and MODE below, then: python nyt_sentiment.py
Year range: 1920–present (auto-detected).
"""

import datetime

# ─── USER SETTINGS ────────────────────────────────────────────────────────────
NYT_API_KEY = "7DdBw359pt8APA0PFXSdRNdoZz1d3eo2ek6AAOI0ak7eYRFz"
MODE        = "both"            # "collect" | "analyze" | "both"
# Option A — inside the repo (recommended)
OUT_DIR = os.path.expanduser("~/Desktop/Code/publichousingnyc/sentiment_output")
YEAR_START  = 1920
YEAR_END    = datetime.date.today().year   # auto-set to current year
# ──────────────────────────────────────────────────────────────────────────────

import os, time, csv
import requests
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

os.makedirs(OUT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. DOMAIN-ADAPTED VADER LEXICON
# ═══════════════════════════════════════════════════════════════════════════════

POSITIVE_TERMS = {
    "modern": 2.0, "sanitary": 2.5, "decent": 2.0, "spacious": 2.0,
    "ventilated": 1.5, "improved": 1.5, "clearance": 1.0,
    "wholesome": 2.0, "adequate": 1.5, "rehabilitation": 1.5,
    "revitalization": 1.5, "relief": 1.5,
    "affordable": 1.5,   # added for post-1970 coverage
    "voucher": 0.5,
}

NEGATIVE_TERMS = {
    "slum": -2.5, "slums": -2.5, "blight": -2.5, "blighted": -2.5,
    "decrepit": -2.0, "vermin": -2.5, "overcrowded": -2.0,
    "overcrowding": -2.0, "dilapidated": -2.0, "squalid": -3.0,
    "squalor": -3.0, "crime-ridden": -2.5, "delinquency": -1.5,
    "delinquent": -1.5, "gang": -2.0, "gangs": -2.0,
    "vandalism": -2.0, "deterioration": -2.0, "deteriorating": -2.0,
    "abandoned": -2.0, "abandonment": -2.0, "neglect": -2.0,
    "neglected": -2.0, "rundown": -2.0,
    "mold": -1.5, "lead": -1.0, "infestation": -2.0,   # post-1990 NYCHA
    "backlog": -1.5, "underfunded": -1.5, "crumbling": -2.0,
}

DRIFT_TERMS = {
    "project":   {"before": 0.0,  "after": -1.0, "cutoff": 1950},
    "projects":  {"before": 0.0,  "after": -1.0, "cutoff": 1950},
    "clearance": {"before": 1.0,  "after": -1.5, "cutoff": 1955},
    "welfare":   {"before": 0.5,  "after": -1.0, "cutoff": 1955},
    "troubled":  {"before": -1.0, "after": -1.5, "cutoff": 1960},
    "voucher":   {"before": 0.0,  "after":  0.5, "cutoff": 1974},
    "pact":      {"before": 0.0,  "after":  0.5, "cutoff": 2016},
}


def build_vader(year):
    a = SentimentIntensityAnalyzer()
    for t, s in POSITIVE_TERMS.items(): a.lexicon[t] = s
    for t, s in NEGATIVE_TERMS.items(): a.lexicon[t] = s
    for t, cfg in DRIFT_TERMS.items():
        a.lexicon[t] = cfg["before"] if year < cfg["cutoff"] else cfg["after"]
    return a


# ═══════════════════════════════════════════════════════════════════════════════
# 2. KEYWORD FILTERS
# ═══════════════════════════════════════════════════════════════════════════════

TIER1_TERMS = [
    "public housing", "housing project", "housing projects",
    "nycha", "housing authority", "slum clearance", "urban renewal",
    "tenement", "tenements",
    "first houses", "vladeck houses", "laguardia houses",
    "la guardia houses", "baruch houses", "alfred e. smith houses",
    "lillian wald houses", "jacob riis houses", "wald houses",
    "riis houses", "smith houses", "gompers houses", "rutgers houses",
    # post-1970
    "section 8", "housing voucher", "housing vouchers",
    "rental assistance demonstration", "pact program",
    "faircloth amendment", "hope vi", "mixed-income housing",
]

TIER2_GEO    = ["lower east side", "east side", "manhattan",
                "new york city", "new york"]
BROAD_TERMS  = {"tenement", "tenements", "housing authority"}
EXCL_SECTION = {"real estate"}
EXCL_MATERIAL= {"letter","letters","correction","corrections",
                "paid notice","classified","obituaries"}


def article_text(a):
    h = a.get("headline", {})
    return " ".join([
        (h.get("main","") if isinstance(h,dict) else h) or "",
        a.get("snippet","") or "",
        a.get("lead_paragraph","") or "",
    ]).lower()


def passes_filter(article, year):
    text = article_text(article)
    if any(e in (article.get("type_of_material") or "").lower()
           for e in EXCL_MATERIAL):
        return False
    matched = [t for t in TIER1_TERMS if t in text]
    if not matched:
        return False
    if all(t in BROAD_TERMS for t in matched):
        if not any(g in text for g in TIER2_GEO):
            return False
    if (article.get("section_name") or "").lower() in EXCL_SECTION:
        if "lower east side" not in text and "east side" not in text:
            return False
    if "urban renewal" in matched and year < 1949:
        return False
    return True


# ═══════════════════════════════════════════════════════════════════════════════
# 3. FRAMING CATEGORIES
# ═══════════════════════════════════════════════════════════════════════════════

FRAMES = {
    "reform":       ["modern","decent","sanitary","relief","better housing",
                     "replaces slums","improved","health","clean","wholesome",
                     "light and air","model housing","affordable",
                     "revitalization","mixed-income","hope vi"],
    "construction": ["units","construction begins","approved","budget",
                     "federal funds","groundbreaking","opens","dedicated",
                     "completed","contract","authorized","appropriation",
                     "renovation","rehabilitation","pact","rad"],
    "conflict":     ["protest","oppose","dispute","segregation","displaced",
                     "community","tenant","strike","integration",
                     "discrimination","opposition","evicted","relocation",
                     "lawsuit","organizing","rally"],
    "pathology":    ["crime","welfare","deteriorat","troubled","gang",
                     "delinquency","drugs","vandalism","violence","shooting",
                     "robbery","arrest","disorder","unsafe","dangerous",
                     "mold","infestation","lead paint","heat outage"],
    "crisis":       ["crisis","failure","collapse","demolish","abandon",
                     "overhaul","bankrupt","shortfall","underfunded",
                     "deteriorated","condemned","scandal","nightmare",
                     "backlog","crumbling","federal monitor"],
}


def assign_frame(text):
    scores = {f: sum(1 for m in ms if m in text) for f, ms in FRAMES.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "construction"


# ═══════════════════════════════════════════════════════════════════════════════
# 4. SCORING
# ═══════════════════════════════════════════════════════════════════════════════

FIELDNAMES = ["id","pub_date","year","headline","section","material",
              "word_count","compound","pos","neg","neu",
              "reform_idx","pathology_idx","frame_ratio","frame"]


def score_article(article, year):
    text = article_text(article)
    vs   = build_vader(year).polarity_scores(text)
    words = text.split(); n = max(len(words), 1)
    ri = sum(1 for w in words if w in POSITIVE_TERMS) / n
    pi = sum(1 for w in words if w in NEGATIVE_TERMS) / n
    d  = ri + pi
    return {
        "id":           article.get("_id",""),
        "pub_date":     article.get("pub_date","")[:10],
        "year":         year,
        "headline":     (article.get("headline") or {}).get("main",""),
        "section":      article.get("section_name",""),
        "material":     article.get("type_of_material",""),
        "word_count":   int(article.get("word_count") or 0),
        "compound":     round(vs["compound"],4),
        "pos":          round(vs["pos"],4),
        "neg":          round(vs["neg"],4),
        "neu":          round(vs["neu"],4),
        "reform_idx":   round(ri,5),
        "pathology_idx":round(pi,5),
        "frame_ratio":  round((ri-pi)/d,4) if d>0 else 0.0,
        "frame":        assign_frame(text),
    }


# ═══════════════════════════════════════════════════════════════════════════════
# 5. DATA COLLECTION
# ═══════════════════════════════════════════════════════════════════════════════

RAW_CSV = os.path.join(OUT_DIR, "nyt_raw_corpus.csv")


def collect():
    seen_ids, rows, errors = set(), [], []
    if os.path.exists(RAW_CSV):
        ex = pd.read_csv(RAW_CSV)
        seen_ids = set(ex["id"].tolist())
        rows     = ex.to_dict("records")
        print(f"  Resuming: {len(rows)} articles already collected.")

    months = (YEAR_END - YEAR_START + 1) * 12
    print(f"  {YEAR_START}–{YEAR_END}: {months} calls "
          f"(≈{months*12//3600:.0f}h {(months*12%3600)//60:.0f}m)")

    today = datetime.date.today()
    for year in range(YEAR_START, YEAR_END + 1):
        for month in range(1, 13):
            if year == today.year and month > today.month:
                break
            url = (f"https://api.nytimes.com/svc/archive/v1/"
                   f"{year}/{month}.json?api-key={NYT_API_KEY}")
            try:
                r = requests.get(url, timeout=30)
                if r.status_code == 429:
                    print(f"  Rate limited {year}-{month:02d}, sleeping 60s…")
                    time.sleep(60)
                    r = requests.get(url, timeout=30)
                r.raise_for_status()
                articles = r.json().get("response",{}).get("docs",[])
            except Exception as e:
                print(f"  ERROR {year}-{month:02d}: {e}")
                errors.append(f"{year}-{month:02d}: {e}")
                time.sleep(12); continue

            mc = 0
            for art in articles:
                aid = art.get("_id","")
                if aid in seen_ids: continue
                if passes_filter(art, year):
                    rows.append(score_article(art, year))
                    seen_ids.add(aid); mc += 1

            print(f"  {year}-{month:02d}: {len(articles):5d} total, "
                  f"{mc:3d} matched  (corpus: {len(rows)})")
            pd.DataFrame(rows, columns=FIELDNAMES).to_csv(RAW_CSV, index=False)
            time.sleep(12)

    print(f"\nDone. {len(rows)} articles.")
    if errors:
        open(os.path.join(OUT_DIR,"collection_errors.txt"),"w").write("\n".join(errors))
        print(f"  {len(errors)} errors logged.")


# ═══════════════════════════════════════════════════════════════════════════════
# 6. ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

NAVY="'#1a3a5c'"; BLUE="#2166ac"; RED="#c0392b"; TEAL="#006464"
GREY="#666666";   LGREY="#cccccc"

FRAME_COLORS = {"reform":"#2166ac","construction":"#aaaaaa",
                "conflict":"#e07b39","pathology":"#c0392b","crisis":"#7b1a1a"}

PERIODS = [
    (1920,1933,"#8c510a","Pre-NYCHA"),
    (1934,1941,"#1a3a5c","I"),
    (1942,1959,"#2166ac","II"),
    (1960,1973,"#4393c3","III"),
    (1974,1999,"#92c5de","IV"),
    (2000,YEAR_END,"#d1e5f0","V"),
]

NYCHA_EVENTS = [
    (1934,"NYCHA\nfounded"),(1935,"First\nHouses"),(1940,"Vladeck"),
    (1949,"Riis+Wald"),(1959,"Baruch"),(1969,"Brooke\nAmend."),
    (1974,"Sec 8\nHCV"),(1994,"HOPE VI"),(2016,"PACT"),
]


def period_shade(ax, alpha=0.07):
    for s,e,c,_ in PERIODS:
        ax.axvspan(s, min(e,YEAR_END), alpha=alpha, color=c, zorder=0)
    for xv in [1934,1942,1960,1974,2000]:
        if xv <= YEAR_END:
            ax.axvline(xv, color=GREY, lw=0.7, ls="--", alpha=0.35)


def bai_perron_breaks(series, max_breaks=5, min_seg=4):
    y = np.array(series); n = len(y)
    def ssr(s): return np.sum((s-s.mean())**2) if len(s) else 0.0
    def one_break(sub):
        best, bp = np.inf, None
        for i in range(min_seg, len(sub)-min_seg+1):
            s = ssr(sub[:i])+ssr(sub[i:])
            if s < best: best, bp = s, i
        return bp, best
    breaks, segs = [], [(0,n)]
    for _ in range(max_breaks):
        best_gain, best_bp = 0, None
        for s,e in segs:
            seg = y[s:e]
            if (e-s) < 2*min_seg: continue
            bp, ns = one_break(seg)
            if bp is None: continue
            gain = ssr(seg)-ns
            if gain > best_gain: best_gain, best_bp = gain, s+bp
        if best_bp is None or best_gain < 1e-6: break
        breaks.append(best_bp); breaks.sort()
        b = [0]+breaks+[n]
        segs = [(b[i],b[i+1]) for i in range(len(b)-1)]
    return sorted(breaks)


def analyze():
    if not os.path.exists(RAW_CSV):
        raise FileNotFoundError(f"No corpus at {RAW_CSV}")

    df = pd.read_csv(RAW_CSV, parse_dates=["pub_date"])
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["year","compound"])
    df = df[(df["year"]>=YEAR_START)&(df["year"]<=YEAR_END)]
    print(f"\n=== Corpus: {len(df):,} articles {YEAR_START}–{YEAR_END} ===")

    def wavg(g):
        wc = g["word_count"].clip(lower=1)
        return np.average(g["compound"], weights=wc)

    annual = (df.groupby("year").apply(wavg)
                .rename("compound_wavg").reset_index())
    annual.columns = ["year","compound_wavg"]
    annual = annual.sort_values("year")
    annual["smooth"] = annual["compound_wavg"].rolling(5,center=True,min_periods=2).mean()
    annual = annual.merge(df.groupby("year").size().rename("n"), on="year")

    ya = annual["year"].values
    break_idxs  = bai_perron_breaks(annual["smooth"].values, max_breaks=5)
    break_years = [int(ya[i]) for i in break_idxs]
    print(f"  Structural breaks: {break_years}")

    frame_decade = (df.assign(decade=(df["year"]//10)*10)
                     .groupby(["decade","frame"]).size().unstack(fill_value=0))
    frame_pct = frame_decade.div(frame_decade.sum(axis=1),axis=0)*100

    # ── Fig 1: Sentiment time series ──────────────────────────────────────────
    fig1,(ax1,ax2)=plt.subplots(2,1,figsize=(18,9),
                                gridspec_kw={"height_ratios":[3,1],"hspace":0.10})
    fig1.patch.set_facecolor("white")

    ax1.bar(annual["year"],annual["compound_wavg"],
            color=[BLUE if v>=0 else RED for v in annual["compound_wavg"]],
            alpha=0.28,width=0.85)
    ax1.plot(annual["year"],annual["smooth"],"-",color="#1a3a5c",lw=2.5,
             label="5-yr rolling mean")
    ax1.axhline(0,color=GREY,lw=0.8)

    for by in break_years:
        ax1.axvline(by,color=RED,lw=1.5,ls="--",alpha=0.85)
        ax1.text(by+0.4,0.94,f"Break\n{by}",
                 transform=ax1.get_xaxis_transform(),
                 fontsize=7,color=RED,va="top",
                 bbox=dict(boxstyle="round,pad=0.2",fc="white",ec=RED,alpha=0.9,lw=0.5))

    for yr,name in NYCHA_EVENTS:
        if YEAR_START<=yr<=YEAR_END:
            ax1.axvline(yr,color=TEAL,lw=0.8,ls=":",alpha=0.55)
            ax1.text(yr+0.3,0.72,name,transform=ax1.get_xaxis_transform(),
                     fontsize=6,color=TEAL,rotation=90,va="top")

    period_shade(ax1)
    for x,lbl in [(1926,"Pre"),(1937,"I"),(1950,"II"),(1966,"III"),(1986,"IV"),(2012,"V")]:
        if YEAR_START<=x<=YEAR_END:
            ax1.text(x,0.04,lbl,transform=ax1.get_xaxis_transform(),
                     ha="center",fontsize=8,color=GREY,style="italic",fontweight="bold")

    ax1.set_ylabel("VADER compound sentiment\n(word-count weighted mean)",fontsize=9.5)
    ax1.set_xlim(YEAR_START-1,YEAR_END+1)
    ax1.legend(fontsize=9,loc="upper right",framealpha=0.92,edgecolor=LGREY)
    ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax1.set_title(f"NYT Sentiment on Public Housing / LES, {YEAR_START}–{YEAR_END}\n"
                  "Domain-adapted VADER  ·  5-yr rolling mean  ·  Bai-Perron breaks",
                  fontsize=11,fontweight="bold",pad=10)
    ax1.tick_params(labelbottom=False)

    ax2.bar(annual["year"],annual["n"],color=GREY,alpha=0.5,width=0.85)
    ax2.set_ylabel("Articles\nper year",fontsize=8.5)
    ax2.set_xlabel("Year",fontsize=9.5)
    ax2.set_xlim(YEAR_START-1,YEAR_END+1)
    period_shade(ax2,alpha=0.04)
    ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

    fig1.savefig(os.path.join(OUT_DIR,"fig_sentiment_timeseries.png"),
                 dpi=300,bbox_inches="tight",facecolor="white")
    print("  Saved: fig_sentiment_timeseries.png")
    plt.close(fig1)

    # ── Fig 2: Frame shares ────────────────────────────────────────────────────
    decades_plot = sorted([d for d in frame_pct.index
                           if YEAR_START<=d<=YEAR_END])
    fig2,ax=plt.subplots(figsize=(max(10,len(decades_plot)*1.4),6))
    fig2.patch.set_facecolor("white")
    bottoms=np.zeros(len(decades_plot)); x=np.arange(len(decades_plot))
    for frame in ["reform","construction","conflict","pathology","crisis"]:
        if frame not in frame_pct.columns: frame_pct[frame]=0.0
        vals=[frame_pct.loc[d,frame] if d in frame_pct.index else 0.0
              for d in decades_plot]
        ax.bar(x,vals,bottom=bottoms,color=FRAME_COLORS[frame],
               label=frame.capitalize(),edgecolor="white",lw=0.6,width=0.6)
        for xi,(v,b) in enumerate(zip(vals,bottoms)):
            if v>8:
                ax.text(xi,b+v/2,f"{v:.0f}%",ha="center",va="center",
                        fontsize=7.5,color="white",fontweight="bold")
        bottoms+=np.array(vals)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{d}s" for d in decades_plot],fontsize=9)
    ax.set_ylabel("Share of articles (%)",fontsize=9.5)
    ax.set_ylim(0,105)
    ax.legend(fontsize=9,loc="upper right",framealpha=0.92,edgecolor=LGREY)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.set_title(f"NYT Framing of Public Housing by Decade, "
                 f"{YEAR_START}s–{YEAR_END//10*10}s\n"
                 "Reform dominant → Construction → Conflict → Pathology/Crisis",
                 fontsize=11,fontweight="bold",pad=10)
    fig2.savefig(os.path.join(OUT_DIR,"fig_frame_shares.png"),
                 dpi=300,bbox_inches="tight",facecolor="white")
    print("  Saved: fig_frame_shares.png")
    plt.close(fig2)

    # ── Fig 3: Sentiment by frame ──────────────────────────────────────────────
    fig3,ax3=plt.subplots(figsize=(18,6))
    fig3.patch.set_facecolor("white")
    fa=(df.groupby(["year","frame"])["compound"].mean().unstack(fill_value=np.nan))
    for frame in ["reform","pathology","crisis","conflict"]:
        if frame not in fa.columns: continue
        vals=fa[frame].rolling(5,center=True,min_periods=2).mean()
        ax3.plot(fa.index,vals,"-",color=FRAME_COLORS[frame],lw=2.0,
                 label=frame.capitalize(),alpha=0.85)
    period_shade(ax3)
    ax3.axhline(0,color=GREY,lw=0.8)
    ax3.set_xlabel("Year",fontsize=9.5)
    ax3.set_ylabel("Mean VADER compound score",fontsize=9.5)
    ax3.set_xlim(YEAR_START-1,YEAR_END+1)
    ax3.legend(fontsize=9,loc="lower left",framealpha=0.92,edgecolor=LGREY)
    ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    ax3.set_title(f"Sentiment by Narrative Frame, {YEAR_START}–{YEAR_END}",
                  fontsize=11,fontweight="bold",pad=10)
    fig3.savefig(os.path.join(OUT_DIR,"fig_sentiment_by_frame.png"),
                 dpi=300,bbox_inches="tight",facecolor="white")
    print("  Saved: fig_sentiment_by_frame.png")
    plt.close(fig3)

    # ── CSVs ──────────────────────────────────────────────────────────────────
    annual.to_csv(os.path.join(OUT_DIR,"annual_sentiment.csv"),index=False)
    frame_pct.to_csv(os.path.join(OUT_DIR,"frame_shares_by_decade.csv"))
    print(f"\nAll outputs in: {os.path.abspath(OUT_DIR)}/")
    print(frame_pct.round(1).to_string())


# ═══════════════════════════════════════════════════════════════════════════════
# 7. ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print(f"Mode={MODE} | Years={YEAR_START}–{YEAR_END}")
    months = (YEAR_END-YEAR_START+1)*12
    print(f"Output: {os.path.abspath(OUT_DIR)}/\n")
    if MODE in ("collect","both"):
        print(f"=== Phase 1: {months} API calls ≈"
              f"{months*12//3600:.0f}h {(months*12%3600)//60:.0f}m ===\n")
        collect()
    if MODE in ("analyze","both"):
        print("\n=== Phase 2: Analysis ===")
        analyze()

Mode=both | Years=1920–2026
Output: /Users/amyxqc/Desktop/Code/publichousingnyc/sentiment_output/

=== Phase 1: 1284 API calls ≈4h 16m ===

  1920–2026: 1284 calls (≈4h 16m)
  1920-01:  7172 total,   1 matched  (corpus: 1)
  1920-02:  7228 total,   2 matched  (corpus: 3)
  1920-03:  7400 total,   0 matched  (corpus: 3)
  1920-04:  7078 total,   2 matched  (corpus: 5)
  1920-05:  7802 total,   1 matched  (corpus: 6)
  1920-06:  7283 total,   3 matched  (corpus: 9)
  1920-07:  7379 total,   0 matched  (corpus: 9)
  1920-08:  7105 total,   6 matched  (corpus: 15)
  Rate limited 1920-09, sleeping 60s…
  ERROR 1920-09: 429 Client Error: Too Many Requests for url: https://api.nytimes.com/svc/archive/v1/1920/9.json?api-key=7DdBw359pt8APA0PFXSdRNdoZz1d3eo2ek6AAOI0ak7eYRFz
  Rate limited 1920-10, sleeping 60s…
  ERROR 1920-10: 429 Client Error: Too Many Requests for url: https://api.nytimes.com/svc/archive/v1/1920/10.json?api-key=7DdBw359pt8APA0PFXSdRNdoZz1d3eo2ek6AAOI0ak7eYRFz
  Rate limited 1

KeyboardInterrupt: 